# Text-to-video experiment (optional)

This notebook is a **small experiment** with Hugging Face `diffusers` and a public text-to-video model (**Zeroscope v2 576w**).

**Expectations:** GPU strongly recommended (VRAM and time). If loading or inference fails on your machine, document what you tried; the graded part is the **image** pipeline in `generate_image_diffusers.py`.

In [ ]:
# Run once if needed
# %pip install -q torch diffusers transformers accelerate safetensors imageio imageio-ffmpeg

In [ ]:
from pathlib import Path

import torch
from diffusers import DiffusionPipeline
from diffusers.utils import export_to_video

MODEL_ID = "cerspense/zeroscope_v2_576w"
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

pipe = DiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=dtype)
if device == "cuda":
    pipe.enable_model_cpu_offload()
    pipe.enable_vae_slicing()
else:
    pipe = pipe.to(device)

In [ ]:
prompt = "a panda riding a bicycle through a sunny park, cinematic"

# Zeroscope v2 576w: native-ish resolution; reduce num_frames if you run out of memory.
result = pipe(
    prompt,
    num_inference_steps=40,
    height=320,
    width=576,
    num_frames=24,
)

frames = result.frames[0]
video_path = OUT_DIR / "zeroscope_clip.mp4"
export_to_video(frames, str(video_path), fps=8)
print("Wrote:", video_path.resolve())

**Tips:** If `from_pretrained` fails, check `diffusers` version and the model card for `cerspense/zeroscope_v2_576w`. On low VRAM, lower `num_frames` (e.g. 12) and `num_inference_steps`.